In [2]:
# --- Standard Python ---
import os
import uuid

# --- Environment variables ---
from dotenv import load_dotenv

# --- Google BigQuery ---
from google.cloud import bigquery

# --- LangChain core ---
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.messages import HumanMessage

# --- LangChain OpenAI ---
from langchain_openai import ChatOpenAI, OpenAIEmbeddings

# --- Document loading & splitting ---
from langchain_community.document_loaders import GCSFileLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter




In [14]:
load_dotenv("env.txt")

# Get credentials
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY")


# Verify credentials exist
if not OPENAI_API_KEY:
    raise ValueError("OPENAI_API_KEY not found in environment")


print("✓ Environment variables loaded")
print(f"  OpenAI API Key: {OPENAI_API_KEY[:20]}...")

✓ Environment variables loaded
  OpenAI API Key: sk-proj-DyCFdOryoZkE...


In [35]:
PROJECT = "project-8661af2e-b3ff-46f2-bc9"
DATASET = "RAG"
TABLE = "customers"

table = bq_client.get_table(f"{PROJECT}.{DATASET}.{TABLE}")

schema_text = ""
for field in table.schema:
    schema_text += f"- {field.name} ({field.field_type})\n"

print(schema_text)


- Index (INTEGER)
- Customer_Id (STRING)
- First_Name (STRING)
- Last_Name (STRING)
- Company (STRING)
- City (STRING)
- Country (STRING)
- Phone 1 (STRING)
- Phone 2 (STRING)
- Email (STRING)
- Subscription_Date (DATE)
- Website (STRING)



In [30]:
intent_prompt = ChatPromptTemplate.from_template("""
You are an intent classifier.

Classify the user's question into ONE of the following intents:

- SQL_CUSTOMERS → Questions about customers, counts, trends, revenue, dates, subscriptions, analytics
- HR_POLICY → Questions about HR policies, leave, holidays, benefits, rules
- UNKNOWN → Anything else

Rules:
- Respond with ONLY one word
- No explanation

User question:
{question}

Intent:
""")


In [36]:
PROJECT = "project-8661af2e-b3ff-46f2-bc9"
DATASET = "RAG"

bq_client = bigquery.Client()

llm = ChatOpenAI(
    model="gpt-4o-mini",
    temperature=0
)

embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small"
)



In [43]:
intent_prompt = ChatPromptTemplate.from_template("""
You are an intent classifier.

Classify the user's question into ONE of the following intents:

- SQL_CUSTOMERS
- HR_POLICY
- UNKNOWN

Respond with ONLY one word.

User question:
{question}

Intent:
""")



def detect_intent(question: str) -> str:
    return intent_chain.invoke({"question": question}).content.strip()



In [53]:
user_question = "give me customer count based on year?"
intent = detect_intent(user_question)

print("Detected intent:", intent)


Detected intent: SQL_CUSTOMERS


In [54]:
# --- Text-to-SQL prompt ---
sql_prompt = ChatPromptTemplate.from_template("""
You are a senior data analyst.
Generate ONLY valid BigQuery SQL.
Do not explain anything.
Do not use markdown.
Only output SQL.

Rules:
- Use fully qualified table name: `{project}.{dataset}.{table}`
- Use BigQuery SQL syntax
- Use LIMIT 100
- Only SELECT queries are allowed

Table schema:
{schema}

User question:
{question}

SQL:
""")

# --- Chain used for SQL generation ---
chain = sql_prompt | llm


In [55]:
if intent == "SQL_CUSTOMERS":

    TABLE = "customers"

    table = bq_client.get_table(f"{PROJECT}.{DATASET}.{TABLE}")

    schema_text = ""
    for field in table.schema:
        schema_text += f"- {field.name} ({field.field_type})\n"

    sql = chain.invoke({
        "project": PROJECT,
        "dataset": DATASET,
        "table": TABLE,
        "schema": schema_text,
        "question": user_question
    }).content

    if not sql.strip().lower().startswith("select"):
        raise ValueError("Unsafe SQL")

    rows = [dict(row) for row in bq_client.query(sql).result()]

    explain_prompt = f"""
    Explain the following query result in simple business language.

    Result:
    {rows}
    """

    explanation = llm.invoke(explain_prompt)
    print(explanation.content)
elif intent == "HR_POLICY":

    TABLE = "doc_embeddings"

    vector_search_sql = f"""
    SELECT
      base.content,
      base.source,
      distance
    FROM VECTOR_SEARCH(
      TABLE `{PROJECT}.{DATASET}.{TABLE}`,
      'embedding',
      (SELECT @query_embedding AS embedding),
      top_k => 5
    )
    ORDER BY distance
    """

    query_embedding = embeddings.embed_query(user_question)

    job_config = bigquery.QueryJobConfig(
        query_parameters=[
            bigquery.ArrayQueryParameter(
                "query_embedding", "FLOAT64", query_embedding
            )
        ]
    )

    docs = list(
        bq_client.query(vector_search_sql, job_config=job_config).result()
    )

    context_text = "\n\n".join([row.content for row in docs])

    rag_prompt = f"""
    You are an HR Policy Assistant.

    Answer ONLY from the context.
    If missing, say: "Information not found in policy".

    Context:
    {context_text}

    Question:
    {user_question}
    """

    answer = llm.invoke([HumanMessage(content=rag_prompt)])
    print(answer.content)
else:
    print("Sorry, I cannot handle this question yet.")


The query result shows the number of customers for a business over three years:

- In **2020**, the business had **426 customers**.
- In **2021**, the number of customers decreased to **404**.
- By **2022**, the customer count dropped significantly to **170**.

In simple terms, the business started with a good number of customers in 2020, but there was a decline in the following years, especially in 2022, where the customer count fell sharply. This could indicate potential issues that need to be addressed to improve customer retention or attract new customers.
